# PBMC 3k scRNA-seq Pipeline (Arrow + Scanpy)

This notebook builds a clean, portfolio-ready single-cell RNA-seq analysis pipeline using a Kaggle dataset and Apache Arrow for efficient data handoff and storage.

Goals:
- Pull or place the Kaggle dataset locally
- Load into AnnData
- Run QC, normalization, dimensionality reduction, clustering
- Export intermediate results with Arrow


In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path(".").resolve()))

import os
import pandas as pd
import numpy as np
import scanpy as sc
import pyarrow as pa
import pyarrow.feather as feather

from src.arrow_sc_utils import (
    ensure_data_dir,
    try_kaggle_download,
    load_dataset_auto,
    write_arrow_qc_table,
)

# Reproducibility
sc.settings.verbosity = 2
sc.settings.set_figure_params(figsize=(6, 4), dpi=120)

DATASET = "mannekuntanagendra/single-cell-rna-seq-analysis-qc-clustering-pbmc-3k"
DATA_DIR = Path("data/pbmc3k")
OUTPUT_DIR = Path("outputs")
ARROW_DIR = OUTPUT_DIR / "arrow"
OUTPUT_DIR.mkdir(exist_ok=True)
ARROW_DIR.mkdir(exist_ok=True)


## Get the Data

If you have Kaggle CLI configured, this will download and unzip the dataset. Otherwise, manually download from Kaggle and place the files inside `data/pbmc3k/`.


In [ ]:
ensure_data_dir(DATA_DIR)

# This will no-op if kaggle is not installed or the token is missing
try_kaggle_download(DATASET, DATA_DIR)

# Quick sanity check for files
sorted([p.name for p in DATA_DIR.rglob("*") if p.is_file()])[:10]


## Load Dataset


In [ ]:
adata = load_dataset_auto(DATA_DIR)

adata


## Basic QC


In [ ]:
# Identify mitochondrial genes (simple heuristic)
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

# Persist QC metrics to Arrow
write_arrow_qc_table(adata, ARROW_DIR / "qc_metrics.feather")

# Visualize
sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")


## Filter, Normalize, Log1p


In [ ]:
# Filters can be tuned; these are conservative defaults
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata = adata[adata.obs.pct_counts_mt < 15].copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)


## Feature Selection and PCA


In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")

sc.pl.pca_variance_ratio(adata, log=True)


## Neighbors, UMAP, Clustering


In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.6)

sc.pl.umap(adata, color=["leiden"])


## Marker Genes


In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="t-test")
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)


## Save Outputs


In [ ]:
adata.write(OUTPUT_DIR / "pbmc3k_processed.h5ad")

# Optionally export expression matrix to Arrow (small datasets only)
expr_df = adata.to_df()
feather.write_feather(pa.Table.from_pandas(expr_df), ARROW_DIR / "expression_matrix.feather")

print("Saved:")
print(OUTPUT_DIR / "pbmc3k_processed.h5ad")
print(ARROW_DIR / "qc_metrics.feather")
print(ARROW_DIR / "expression_matrix.feather")


## Portfolio Notes

Suggested talking points for your GitHub/LinkedIn portfolio:
- Built an end-to-end single-cell pipeline using Scanpy and AnnData
- Integrated Apache Arrow for fast, portable intermediate artifacts
- Designed reproducible steps: QC, normalization, dimensionality reduction, clustering, marker gene discovery
